**Autor:** Alan Sastre  
**GitHub:** [github.com/alansastre/genai-agentes](https://github.com/alansastre/genai-agentes)

---

# Integración de MCP en agentes LangChain

El **Model Context Protocol** (MCP) es un protocolo abierto que estandariza la forma en que las aplicaciones proporcionan herramientas y contexto a los modelos de lenguaje. En lugar de implementar integraciones personalizadas para cada servicio externo, MCP permite conectar agentes a **servidores de herramientas estandarizados** que exponen capacidades específicas.

La ventaja principal de usar MCP es que existe un ecosistema creciente de **servidores públicos** mantenidos por organizaciones como Microsoft, Stripe, MongoDB, Notion y muchas otras. Esto significa que un agente LangChain puede acceder a documentación actualizada, bases de datos, APIs y servicios sin necesidad de escribir código de integración.

> MCP actúa como un **puente universal** entre agentes y servicios externos. Un servidor MCP expone herramientas (tools), recursos (resources) y prompts que cualquier cliente compatible puede consumir.

```mermaid
flowchart LR
    A[Agente LangChain] --> B[langchain-mcp-adapters]
    B --> C[MCP Server 1]
    B --> D[MCP Server 2]
    B --> E[MCP Server N]
    C --> F[Docs Microsoft]
    D --> G[Base de datos]
    E --> H[APIs externas]
```

## Instalación de langchain-mcp-adapters

Para conectar agentes LangChain con servidores MCP se utiliza la biblioteca `langchain-mcp-adapters`. Esta biblioteca proporciona un cliente que gestiona las conexiones y convierte las herramientas MCP al formato compatible con LangChain.

```bash .noeval
pip install langchain-mcp-adapters
```

El componente principal es `MultiServerMCPClient`, que permite conectar con **uno o varios servidores MCP** simultáneamente. Cada servidor se configura indicando su tipo de transporte y la URL o comando correspondiente.

## Conexión con servidores MCP públicos

Los servidores MCP pueden funcionar mediante dos tipos de transporte principales:

* **HTTP** (`"http"`): El servidor está accesible en una URL pública. No requiere instalación local. La librería también acepta el alias `"streamable_http"` para este transporte.
* **stdio**: El servidor se ejecuta como un subproceso local y la comunicación se realiza mediante entrada/salida estándar.

Para servidores públicos como el de **Microsoft Learn**, se utiliza transporte HTTP, lo que significa que no necesitamos instalar ni ejecutar nada localmente. El servidor ya está disponible en internet.

> El registro de servidores MCP públicos está disponible en [github.com/mcp](https://github.com/mcp). Allí se pueden encontrar integraciones oficiales para GitHub, Notion, MongoDB, Stripe y muchos otros servicios.

## Ejemplo práctico con Microsoft Learn MCP

El servidor MCP de **Microsoft Learn** proporciona acceso a la documentación oficial de Microsoft. Incluye herramientas para buscar documentación, obtener páginas completas en markdown y buscar ejemplos de código.

Las herramientas disponibles en este servidor son:

| Herramienta | Descripción |
|-------------|-------------|
| `microsoft_docs_search` | Búsqueda semántica en documentación técnica de Microsoft |
| `microsoft_docs_fetch` | Obtiene una página de documentación en formato markdown |
| `microsoft_code_sample_search` | Busca snippets de código oficiales de Microsoft/Azure |

### Configuración del cliente MCP

La configuración de `MultiServerMCPClient` se realiza mediante un diccionario donde cada clave es un nombre identificador del servidor y el valor contiene los parámetros de conexión. El parámetro `transport` indica el protocolo de comunicación y `url` especifica el endpoint del servidor MCP.

In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "microsoft-learn": {
            "transport": "http",
            "url": "https://learn.microsoft.com/api/mcp"
        }
    }
)

### Obtener herramientas del servidor MCP

La función `get_tools()` es **asíncrona** porque establece conexión con el servidor MCP remoto. Devuelve una lista de herramientas que pueden pasarse directamente a `create_agent`.

In [2]:
tools = await client.get_tools()

print("Herramientas MCP disponibles:")
for tool in tools:
    print(f"  - {tool.name}: {tool.description[:80]}...")

Herramientas MCP disponibles:
  - microsoft_docs_search: Search official Microsoft/Azure documentation to find the most relevant and trus...
  - microsoft_code_sample_search: Search for code snippets and examples in official Microsoft Learn documentation....
  - microsoft_docs_fetch: Fetch and convert a Microsoft Learn documentation webpage to markdown format. Th...


### Crear el agente con herramientas MCP

Una vez obtenidas las herramientas, se crea el agente pasándolas como parámetro.

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    system_prompt="Eres un asistente experto en tecnologías Microsoft y Azure. Usa las herramientas disponibles para buscar información actualizada."
)

## Verificar si el agente usó el MCP

Para comprobar que el agente efectivamente utiliza el servidor MCP, podemos hacerle una pregunta específica sobre un tema de la documentación de Microsoft. Una buena prueba es preguntar sobre **Azure for Students**, ya que es información que el modelo no tendría actualizada en sus datos de entrenamiento.

Cuando un agente usa una herramienta, genera mensajes de tipo `AIMessage` con `tool_calls` y posteriormente `ToolMessage` con el resultado de la llamada. Inspeccionando estos mensajes podemos confirmar el uso del MCP.

In [4]:
from langchain.messages import ToolMessage, AIMessage

# Pregunta específica sobre Azure for Students
pregunta = """
¿Cuánto crédito en dólares incluye la suscripción Azure for Students 
y cuánto tiempo dura? Busca en la documentación oficial de Microsoft.
"""

response = await agent.ainvoke({
    "messages": [{"role": "user", "content": pregunta}]
})

### Análisis del uso de herramientas

La siguiente función analiza la respuesta para determinar si el agente utilizó las herramientas MCP y cuáles fueron invocadas.

In [5]:
def analizar_uso_mcp(response):
    """Analiza si el agente utilizó herramientas MCP."""
    messages = response["messages"]
    
    tool_calls = []
    tool_results = []
    
    for msg in messages:
        # Detectar llamadas a herramientas en mensajes del modelo
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for call in msg.tool_calls:
                tool_calls.append(call["name"])
        
        # Detectar resultados de herramientas
        if isinstance(msg, ToolMessage):
            tool_results.append({
                "tool": msg.name,
                "content_preview": msg.content[:200] + "..." if len(msg.content) > 200 else msg.content
            })
    
    return {
        "uso_mcp": len(tool_calls) > 0,
        "herramientas_usadas": tool_calls,
        "resultados": tool_results
    }

analisis = analizar_uso_mcp(response)

print(f"¿Se usó MCP?: {analisis['uso_mcp']}")
print(f"Herramientas invocadas: {analisis['herramientas_usadas']}")
print(f"\nRespuesta del agente:")
print(response["messages"][-1].content)

¿Se usó MCP?: True
Herramientas invocadas: ['microsoft_docs_search', 'microsoft_docs_fetch', 'microsoft_docs_fetch']

Respuesta del agente:
Respuesta corta:
- Crédito: 100 USD de crédito.
- Duración: 12 meses (1 año).

Notas útiles:
- El crédito se aplica a los cargos durante ese periodo; puedes ver el crédito restante en la Azure Sponsorships portal.
- Al finalizar 12 meses o al agotarse el crédito, la suscripción se cancela a menos que la renueves. Si sigues siendo estudiante activo, puedes renovar cada año para obtener otros 100 USD de crédito.

Fuentes oficiales de Microsoft:
- Azure for Students (Azure Education Hub) – "Azure for Students gives you a $100 credit for 12 months." https://learn.microsoft.com/azure/education-hub/about-azure-for-students
- What is Azure for Students? – resumen de la oferta ($100 de crédito, válido por un año, renovación anual) https://learn.microsoft.com/azure/education-hub/azure-dev-tools-teaching/program-faq#can-i-get-azure-for-students-again-next-ye

Si el agente utilizó correctamente el servidor MCP, veremos que `uso_mcp` es `True` y en `herramientas_usadas` aparecerán nombres como `microsoft_docs_search` o `microsoft_docs_fetch`.